# Trabalho de Séries Temporais — Grupo 5 (PLS)

O notebook tem 3 partes:

1. **Bases** — trata cada base e coloca na lista `bases` (é aqui que se adiciona base nova)
2. **Pipeline** — features, otimização e walk-forward
3. **Comparação** — MAE, ranking, vitórias, resíduos e importância das features

A parte 2 não compara nada: ela só produz as previsões fora da amostra de
5 bases × 4 modelos. A comparação é a parte 3.

Duas regras valem no notebook inteiro: `random_state=42` em tudo que sorteia,
e separação treino/teste **70-30**.

In [18]:
import os
# threads do BLAS em 1: quem paraleliza aqui e o joblib, nao a algebra linear
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import ast
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf

from sklearn.ensemble import RandomForestRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42     # regra do grupo: vale para tudo que sorteia
P_TESTE = 0.30        # regra do grupo: separacao treino/teste 70-30

np.random.seed(RANDOM_STATE)

# 1. Bases

Cada base entra como um dicionário. O `df` precisa chegar aqui **já tratado**:
índice de data, frequência regular, sem furo no alvo.

As externas vão em duas listas, e isso muda como o pipeline as usa:

- `exog_conhecidas` — o valor futuro já é sabido na data da previsão
  (feriado, calendário, promoção planejada). Entra com o valor da própria data.
- `exog_defasadas` — não se sabe o valor futuro (temperatura observada,
  indicador divulgado com atraso). Entra só defasada em `h`.

O 70-30 é aplicado duas vezes: 30% final da série é o **teste**, e 30% final do
que sobrou é a **validação**, onde os hiperparâmetros são escolhidos.

In [29]:
bases = []

def add_base(nome, df, alvo, h, exog_conhecidas=(), exog_defasadas=(),
             n_validacao=None, n_teste=None):
    n = len(df)
    n_teste = n_teste or round(P_TESTE * n)                  # 30% final -> teste
    n_treino = n - n_teste
    n_validacao = n_validacao or round(P_TESTE * n_treino)   # 30% do treino -> validacao

    base = {
        "nome": nome,
        "df": df.sort_index(),
        "alvo": alvo,
        "m": sugerir_m(df[alvo]),                  # periodo sazonal
        "h": h,                                    # horizonte de previsao
        "exog_conhecidas": list(exog_conhecidas),
        "exog_defasadas": list(exog_defasadas),
        "n_validacao": n_validacao,
        "n_teste": n_teste,
    }
    bases.append(base)
    return base

Para escolher o `m` de uma base, a força da sazonalidade da STL — o mesmo
critério da Tarefa 03. Roda antes do `add_base`, só para decidir o número.

In [30]:
def forca_sazonalidade(serie, m):
    res = STL(serie, period=m, robust=True).fit()
    return 1 - np.var(res.resid) / np.var(res.seasonal + res.resid)

def sugerir_m(serie, candidatos=range(2, 101), min_ciclos=3, top=10):
    forcas = []
    for m in candidatos:
        if len(serie) / m < min_ciclos:
            continue
        try:
            forcas.append((m, forca_sazonalidade(serie, m)))
        except Exception:
            continue

    forcas.sort(key=lambda x: -x[1])
    for m, f in forcas[:top]:
        print(f"m={m:>3} | forca={f:.4f}")
    return forcas[0][0]

## Base 1 — vendas semanais

Duas externas: `feriado` (conhecido de antemão) e `temperatura` (só se sabe
depois de acontecer, então entra defasada).

**Para adicionar base nova: copia esta célula, troca a leitura e o `add_base`.**
O resto do notebook roda sozinho em cima da lista `bases`.

In [ ]:
df = pd.read_excel("dados/dados_aula08_sarimax.xlsx")
df["data"] = pd.to_datetime(df["data"])
df = df.set_index("data").asfreq("W-MON")

add_base(
    nome="vendas_semanais",
    df=df,
    alvo="vendas",
    h=4,
    exog_conhecidas=["feriado"],
    exog_defasadas=["temperatura"],
)

df.head()

,vendas,temperatura,feriado
data,,,
2020-01-06,458.431716,20.993428,1
2020-01-13,689.511394,20.928838,0
2020-01-20,610.337570,23.688534,0
2020-01-27,646.711686,26.592109,0
2020-02-03,644.334682,24.178925,0


In [22]:
pd.DataFrame([{
    "base": b["nome"],
    "obs": len(b["df"]),
    "inicio": b["df"].index.min().date(),
    "fim": b["df"].index.max().date(),
    "freq": b["df"].index.freqstr,
    "m": b["m"],
    "h": b["h"],
    "treino": len(b["df"]) - b["n_teste"] - b["n_validacao"],
    "validacao": b["n_validacao"],
    "teste": b["n_teste"],
    "conhecidas": ", ".join(b["exog_conhecidas"]) or "-",
    "defasadas": ", ".join(b["exog_defasadas"]) or "-",
} for b in bases])

,base,obs,inicio,fim,freq,m,h,treino,validacao,teste,conhecidas,defasadas
0,vendas_semanais,208,2020-01-06,2023-12-25,W-MON,52,4,102,44,62,feriado,temperatura


# 2. Pipeline

O que é comum a todos: features, exógenas, origens e o walk-forward.
Depois, um bloco por modelo — cada um com sua função de previsão e sua
função de otimização.

## 2.1 Features

Regra da seção inteira: uma linha com data `t` só pode usar informação de até
`t - h`. Por isso todo lag é `>= h` e todo `rolling` leva `shift(h)` antes.

A mesma tabela vai para o Random Forest e para o PLS.

In [23]:
def criar_features(base, janelas=(4, 8, 12)):
    h, m = base["h"], base["m"]
    y = base["df"][base["alvo"]]

    tab = pd.DataFrame(index=base["df"].index)
    tab["y"] = y

    # lags: recentes (nivel atual) e sazonais (mesmo ponto do ciclo anterior)
    for lag in sorted({h, h + 1, h + 2, h + 3, m, m + h}):
        tab[f"lag_{lag}"] = y.shift(lag)

    # janelas moveis: o shift(h) antes do rolling e o que evita vazamento
    passado = y.shift(h)
    for j in janelas:
        tab[f"media_{j}"] = passado.rolling(j).mean()
        tab[f"desvio_{j}"] = passado.rolling(j).std()
    tab["delta_nivel"] = tab[f"media_{janelas[0]}"] - tab[f"media_{janelas[-1]}"]

    # calendario: vem do indice, entao e conhecido para qualquer data futura
    idx = base["df"].index
    tab["mes"] = idx.month
    tab["dia_semana"] = idx.dayofweek
    tab["semana"] = idx.isocalendar().week.astype(int).to_numpy()

    # encoding ciclico: dezembro e janeiro viram vizinhos, domingo e segunda tambem
    for col, periodo in [("mes", 12), ("dia_semana", 7), ("semana", 52)]:
        tab[f"{col}_sin"] = np.sin(2 * np.pi * tab[col] / periodo)
        tab[f"{col}_cos"] = np.cos(2 * np.pi * tab[col] / periodo)

    # externas
    for col in base["exog_conhecidas"]:
        tab[col] = base["df"][col]
    for col in base["exog_defasadas"]:
        tab[f"{col}_lag_{h}"] = base["df"][col].shift(h)

    # os lags criam NaN so no comeco da serie: essas linhas sao descartadas
    tab = tab.dropna()

    # coluna constante nao informa nada e quebra a padronizacao do PLS
    constantes = [c for c in tab.columns if c != "y" and tab[c].nunique() <= 1]
    return tab.drop(columns=constantes)

## 2.2 Exógenas do SARIMAX

Mesma regra de disponibilidade, no formato que o `statsmodels` consome.

In [24]:
def criar_exog(base):
    if not base["exog_conhecidas"] and not base["exog_defasadas"]:
        return None

    exog = pd.DataFrame(index=base["df"].index)
    for col in base["exog_conhecidas"]:
        exog[col] = base["df"][col]
    for col in base["exog_defasadas"]:
        exog[f"{col}_lag_{base['h']}"] = base["df"][col].shift(base["h"])

    return exog.bfill()

## 2.3 Origens de previsão

O 70-30 da regra do grupo, aplicado duas vezes:

```
[------- treino -------][-- validacao --][------ teste ------]
|<----------- 70% ---------------------->|<------ 30% ------>|
```

A validação escolhe os hiperparâmetros. O teste só é tocado no final.
Os 4 modelos recebem exatamente as mesmas origens.

In [25]:
def origens(base, etapa):
    idx = base["df"].index
    n, h = len(idx), base["h"]

    ini_teste = n - base["n_teste"]
    ini_val = ini_teste - base["n_validacao"]

    a, b = (ini_val, ini_teste) if etapa == "validacao" else (ini_teste, n)

    # passo = h -> blocos de previsao que nao se sobrepoem
    # a origem e a ultima data que o modelo enxerga
    return [idx[p] for p in range(a - 1, b - h, h)]

def datas_futuras(base, origem):
    idx = base["df"].index
    return idx[idx.get_loc(origem) + 1:][:base["h"]]

## 2.4 Walk-forward

Anda pelas origens, chama a função de previsão do modelo em cada uma e junta
tudo num DataFrame longo. É a única parte genérica: recebe `prever` como
argumento e não sabe qual modelo está rodando.

In [26]:
def walk_forward(prever, params, base, tab, exog, lista_origens):
    linhas = []

    for origem in lista_origens:
        datas = datas_futuras(base, origem)
        pred = prever(params, base, tab, exog, origem)

        for passo, (data, valor) in enumerate(zip(datas, pred), start=1):
            linhas.append({
                "base": base["nome"],
                "origem": origem,
                "data": data,
                "passo": passo,
                "y_real": base["df"][base["alvo"]].loc[data],
                "y_previsto": float(valor),
            })

    saida = pd.DataFrame(linhas)
    saida["residuo"] = saida["y_real"] - saida["y_previsto"]
    return saida


def mae_validacao(prever, params, base, tab, exog):
    # roda o mesmo walk-forward na janela de validacao e devolve o MAE
    try:
        prev = walk_forward(prever, params, base, tab, exog, origens(base, "validacao"))
        return mean_absolute_error(prev["y_real"], prev["y_previsto"])
    except Exception:
        return np.inf

## 2.5 SARIMAX

As ordens não são fixas: `d` e `D` saem dos testes de estacionariedade e o
resto é grade `(p,d,q)×(P,D,Q,m)`, em paralelo com `joblib` — mesmo
procedimento da Tarefa 03.

A decisão usa **BIC e MAE juntos**, porque os dois medem coisas diferentes:
BIC é ajuste dentro da amostra penalizado por complexidade, MAE de validação é
erro fora da amostra. BIC baixo não garante MAE baixo. Os dois são normalizados
em 0–1 (min-max) e somados; vence o menor total.

Em duas etapas, por custo: o BIC roda na grade inteira (é um ajuste por
combinação), e o MAE de validação só nas `top_bic` melhores — cada uma custa um
walk-forward completo. A normalização é feita dentro dessa lista final, que é o
conjunto entre o qual se está de fato decidindo.

A etapa do BIC fica em cache. As ordens escolhidas ficam congeladas no
walk-forward de teste.

In [27]:
def prever_sarimax(params, base, tab, exog, origem):
    y_treino = base["df"][base["alvo"]].loc[:origem]
    datas = datas_futuras(base, origem)

    ajuste = SARIMAX(
        y_treino,
        exog=None if exog is None else exog.loc[:origem],
        order=params["order"],
        seasonal_order=params["seasonal_order"],
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False, maxiter=100)

    previsao = ajuste.get_forecast(
        steps=base["h"], exog=None if exog is None else exog.loc[datas]
    )
    return previsao.predicted_mean.to_numpy()

In [28]:
def eh_estacionaria(serie):
    # ADF: p < 0.05 -> estacionaria | KPSS: p > 0.05 -> estacionaria
    return adfuller(serie)[1] < 0.05 and kpss(serie)[1] > 0.05

def sugerir_d(serie, max_d=2):
    s = serie.dropna()
    for d in range(max_d + 1):
        if eh_estacionaria(s):
            return d
        s = s.diff().dropna()
    return max_d

def sugerir_D(serie, m, d):
    s = serie.diff(d).dropna() if d > 0 else serie
    s = s.diff(m).dropna()
    if len(s) < 2 * m:
        return 0
    return 1 if eh_estacionaria(s) else 0

In [ ]:
def _bic_sarimax(order, seasonal_order, y, exog):
    try:
        ajuste = SARIMAX(
            y, exog=exog, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False,
        ).fit(disp=False, maxiter=100)
        return ajuste.bic
    except Exception:
        return None


def normalizar(s):
    # min-max em 0-1; se todos forem iguais, todos valem 0
    faixa = s.max() - s.min()
    return pd.Series(0.0, index=s.index) if faixa == 0 else (s - s.min()) / faixa


def otimizar_sarimax(base, tab, exog, top_bic=30, usar_cache=True):
    # corte: nada do conjunto de teste entra na escolha das ordens
    corte = base["df"].index[-base["n_teste"] - 1]
    y = base["df"][base["alvo"]].loc[:corte]
    ex = None if exog is None else exog.loc[:corte]
    m = base["m"]

    d_sug = sugerir_d(y)
    D_sug = sugerir_D(y, m, d_sug)
    print(f"      d sugerido = {d_sug} | D sugerido = {D_sug} | m = {m}")

    combos = [
        ((p, d, q), (P, D, Q, m))
        for p in range(3) for d in range(d_sug + 1) for q in range(3)
        for P in range(3) for D in range(D_sug + 1) for Q in range(3)
    ]

    # --- etapa 1: BIC na grade inteira (um ajuste por combinacao) ---------
    cache = f"resultados/cache_sarimax_{base['nome']}.csv"
    if usar_cache and os.path.exists(cache):
        grade = pd.read_csv(cache)
        grade["params"] = grade["params"].apply(ast.literal_eval)
    else:
        print(f"      BIC de {len(combos)} combinacoes em paralelo...")
        bics = Parallel(n_jobs=-1, backend="loky")(
            delayed(_bic_sarimax)(order, so, y, ex) for order, so in combos
        )
        grade = pd.DataFrame([
            {"params": {"order": order, "seasonal_order": so}, "bic": bic}
            for (order, so), bic in zip(combos, bics) if bic is not None
        ]).sort_values("bic").reset_index(drop=True)

        grade.assign(params=grade["params"].astype(str)).to_csv(cache, index=False)

    # --- etapa 2: MAE de validacao nas melhores por BIC -------------------
    busca = grade.head(top_bic).reset_index(drop=True)
    print(f"      MAE de validacao nas {len(busca)} melhores por BIC...")

    busca["mae_validacao"] = Parallel(n_jobs=-1, backend="loky")(
        delayed(mae_validacao)(prever_sarimax, p, base, tab, exog) for p in busca["params"]
    )

    # --- decisao: soma dos dois normalizados ------------------------------
    busca["bic_norm"] = normalizar(busca["bic"])
    busca["mae_norm"] = normalizar(busca["mae_validacao"])
    busca["criterio"] = "BIC+MAE normalizados"
    busca["valor"] = busca["bic_norm"] + busca["mae_norm"]

    busca = busca.sort_values("valor").reset_index(drop=True)
    return busca.loc[0, "params"], busca

## 2.6 Holt-Winters

Referência univariada: não recebe exógenas. A grade cruza tendência,
sazonalidade e amortecimento, e a escolha é pelo MAE na validação.

In [ ]:
def prever_holtwinters(params, base, tab, exog, origem):
    y_treino = base["df"][base["alvo"]].loc[:origem]

    ajuste = ExponentialSmoothing(
        y_treino,
        trend=params["trend"],
        seasonal=params["seasonal"],
        damped_trend=params["damped"],
        seasonal_periods=base["m"] if params["seasonal"] else None,
        initialization_method="estimated",
    ).fit(optimized=True)

    return np.asarray(ajuste.forecast(base["h"]))


def otimizar_holtwinters(base, tab, exog):
    grade = [
        {"trend": t, "seasonal": s, "damped": d}
        for t in ("add", None)
        for s in ("add", "mul", None)
        for d in (True, False)
        if not (t is None and d)          # nao existe amortecimento sem tendencia
    ]

    # Holt-Winters nao usa exogenas: passa None
    maes = Parallel(n_jobs=-1, backend="loky")(
        delayed(mae_validacao)(prever_holtwinters, p, base, tab, None) for p in grade
    )

    busca = pd.DataFrame([
        {"params": p, "criterio": "MAE validacao", "valor": e} for p, e in zip(grade, maes)
    ]).sort_values("valor").reset_index(drop=True)

    return busca.loc[0, "params"], busca

## 2.7 Random Forest

Usa a tabela de features. `n_jobs=1` no estimador de propósito: quem paraleliza
é a grade, e aninhar os dois deixa mais lento.

In [ ]:
def prever_rf(params, base, tab, exog, origem):
    treino = tab.loc[:origem]
    futuro = tab.loc[datas_futuras(base, origem)]

    reg = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1, **params)
    reg.fit(treino.drop(columns="y"), treino["y"])

    return reg.predict(futuro.drop(columns="y"))


def otimizar_rf(base, tab, exog):
    grade = [
        {"n_estimators": n, "max_depth": d, "max_features": f,
         "min_samples_split": s, "min_samples_leaf": l}
        for n in (300, 600)
        for d in (None, 6, 12)
        for f in ("sqrt", 0.5)
        for s in (2, 5)
        for l in (1, 2)
    ]

    maes = Parallel(n_jobs=-1, backend="loky")(
        delayed(mae_validacao)(prever_rf, p, base, tab, exog) for p in grade
    )

    busca = pd.DataFrame([
        {"params": p, "criterio": "MAE validacao", "valor": e} for p, e in zip(grade, maes)
    ]).sort_values("valor").reset_index(drop=True)

    return busca.loc[0, "params"], busca

## 2.8 PLS — modelo de especialização do grupo

Mesma tabela de features do Random Forest, para a comparação não ser decidida
por dados diferentes.

O `StandardScaler` não é opcional: o PLS constrói componentes maximizando
covariância com o alvo, então sem padronizar a feature de maior escala domina
tudo. `n_components` é o hiperparâmetro principal — poucos componentes
subajustam, muitos fazem o modelo voltar a ser uma regressão linear comum.

In [ ]:
def prever_pls(params, base, tab, exog, origem):
    treino = tab.loc[:origem]
    futuro = tab.loc[datas_futuras(base, origem)]

    reg = make_pipeline(StandardScaler(), PLSRegression(n_components=params["n_components"]))
    reg.fit(treino.drop(columns="y"), treino["y"])

    return np.asarray(reg.predict(futuro.drop(columns="y"))).ravel()


def otimizar_pls(base, tab, exog):
    # o limite e o numero de features disponiveis
    n_max = min(15, tab.shape[1] - 1)
    grade = [{"n_components": k} for k in range(1, n_max + 1)]

    maes = Parallel(n_jobs=-1, backend="loky")(
        delayed(mae_validacao)(prever_pls, p, base, tab, exog) for p in grade
    )

    busca = pd.DataFrame([
        {"params": p, "criterio": "MAE validacao", "valor": e} for p, e in zip(grade, maes)
    ]).sort_values("valor").reset_index(drop=True)

    return busca.loc[0, "params"], busca

## 2.9 Rodar

Para cada base e cada modelo: otimiza fora do teste → congela os
hiperparâmetros → walk-forward no teste.

In [ ]:
MODELOS = [
    ("SARIMAX",       otimizar_sarimax,      prever_sarimax),
    ("Holt-Winters",  otimizar_holtwinters,  prever_holtwinters),
    ("Random Forest", otimizar_rf,           prever_rf),
    ("PLS",           otimizar_pls,          prever_pls),
]


def rodar(bases, modelos=MODELOS):
    previsoes, escolhidos, buscas = [], [], []

    for base in bases:
        tab = criar_features(base)
        exog = criar_exog(base)
        org = origens(base, "teste")

        print(f"[{base['nome']}] {len(base['df'])} obs | h={base['h']} | m={base['m']} | "
              f"{tab.shape[1] - 1} features | {len(org)} origens de teste")

        for nome, otimizar, prever in modelos:
            inicio = pd.Timestamp.now()
            print(f"   {nome}")

            params, busca = otimizar(base, tab, exog)

            # hiperparametros congelados: daqui pra frente ninguem mais mexe
            prev = walk_forward(prever, params, base, tab, exog, org)
            prev.insert(1, "modelo", nome)

            segundos = (pd.Timestamp.now() - inicio).total_seconds()
            print(f"      {params}  ({segundos:.0f}s)")

            previsoes.append(prev)
            busca["base"], busca["modelo"] = base["nome"], nome
            buscas.append(busca)
            escolhidos.append({
                "base": base["nome"],
                "modelo": nome,
                "params": str(params),
                "criterio": busca.loc[0, "criterio"],
                "valor": busca.loc[0, "valor"],
                "segundos": round(segundos, 1),
            })

    return (pd.concat(previsoes, ignore_index=True),
            pd.DataFrame(escolhidos),
            pd.concat(buscas, ignore_index=True))

In [ ]:
previsoes, parametros, buscas = rodar(bases)

previsoes.to_csv("resultados/previsoes.csv", index=False)
parametros.to_csv("resultados/hiperparametros.csv", index=False)
buscas.assign(params=buscas["params"].astype(str)).to_csv("resultados/busca.csv", index=False)

parametros

# 3. Comparação

Tudo daqui pra baixo sai de `previsoes` — as previsões fora da amostra, com os
hiperparâmetros já congelados. Nada aqui é específico de base ou de modelo:
vale igual para as 5 bases e os 4 modelos.

## 3.1 MAE e ranking dentro de cada base

Bases com escalas diferentes não podem ter o MAE somado nem promediado entre
si. Por isso o MAE e o ranking são calculados **dentro** de cada base, e a
comparação entre bases é feita depois pela posição, não pelo valor.

In [ ]:
previsoes["erro_abs"] = (previsoes["y_real"] - previsoes["y_previsto"]).abs()

mae = (previsoes.groupby(["base", "modelo"])["erro_abs"]
       .mean()
       .rename("MAE")
       .reset_index())

# posicao 1 = menor MAE dentro da base
mae["posicao"] = mae.groupby("base")["MAE"].rank(method="min").astype(int)

mae.sort_values(["base", "posicao"])

In [ ]:
tabela_mae = mae.pivot(index="modelo", columns="base", values="MAE").round(3)
tabela_posicao = mae.pivot(index="modelo", columns="base", values="posicao")

print("MAE por base")
print(tabela_mae.to_string())
print()
print("Posicao por base")
print(tabela_posicao.to_string())

## 3.2 Melhor modelo em cada base

In [ ]:
melhores = (mae.loc[mae.groupby("base")["MAE"].idxmin()]
            .set_index("base")[["modelo", "MAE"]]
            .rename(columns={"modelo": "melhor_modelo", "MAE": "melhor_MAE"}))

melhores

## 3.3 Vitórias e posição média

Vitória = ter o menor MAE da base. A posição média resume o desempenho geral
do modelo sem misturar escalas — é o número que permite comparar as 5 bases.

In [ ]:
placar = (mae.groupby("modelo")
          .agg(vitorias=("posicao", lambda p: int((p == 1).sum())),
               posicao_media=("posicao", "mean"),
               melhor_posicao=("posicao", "min"),
               pior_posicao=("posicao", "max"),
               bases=("base", "nunique"))
          .sort_values(["vitorias", "posicao_media"], ascending=[False, True]))

placar["posicao_media"] = placar["posicao_media"].round(2)
placar

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

axes[0].barh(placar.index, placar["vitorias"], color="steelblue")
axes[0].set_xlabel("vitorias (menor MAE na base)")
axes[0].set_title("Vitorias por modelo", loc="left")

axes[1].barh(placar.index, placar["posicao_media"], color="darkorange")
axes[1].set_xlabel("posicao media (menor e melhor)")
axes[1].set_title("Posicao media", loc="left")

for ax in axes:
    ax.invert_yaxis()
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)

plt.tight_layout()
plt.show()

## 3.4 Resíduos

A tabela de Ljung-Box sai para as 20 combinações — é ela que vai no corpo do
relatório. A análise gráfica detalhada vem depois, do **melhor modelo de cada
base**, que é onde a discussão de viés e padrão remanescente importa.

Como o walk-forward usa passo `= h`, os blocos de previsão não se sobrepõem e
os resíduos do teste formam uma série contínua no tempo — é isso que torna a
ACF interpretável aqui.

In [ ]:
def ljung_box(residuos, lags=10):
    lags = max(1, min(lags, len(residuos) // 5))
    r = acorr_ljungbox(residuos, lags=[lags], return_df=True)
    return lags, float(r["lb_stat"].iloc[0]), float(r["lb_pvalue"].iloc[0])


linhas = []
for (b, mdl), g in previsoes.groupby(["base", "modelo"]):
    residuos = g.sort_values("data")["residuo"]
    lags, stat, p = ljung_box(residuos)
    linhas.append({
        "base": b,
        "modelo": mdl,
        "n": len(residuos),
        "lags": lags,
        "vies": residuos.mean(),          # media do residuo: > 0 subestima, < 0 superestima
        "desvio": residuos.std(),
        "lb_stat": stat,
        "p_valor": p,
        "ruido_branco": "sim" if p > 0.05 else "nao",
    })

tabela_ljung = pd.DataFrame(linhas).sort_values(["base", "modelo"]).round(4)
tabela_ljung.to_csv("resultados/ljung_box.csv", index=False)

tabela_ljung

In [ ]:
def analisar_residuos(base_nome, modelo_nome):
    g = (previsoes.query("base == @base_nome and modelo == @modelo_nome")
         .sort_values("data"))
    residuos = g.set_index("data")["residuo"]
    lags, stat, p = ljung_box(residuos)

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

    # residuos no tempo: procura tendencia, viés e mudanca de variancia
    axes[0].plot(residuos.index, residuos.values, color="steelblue",
                 marker="o", markersize=3, linewidth=1)
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].axhline(residuos.mean(), color="tomato", linestyle="--", linewidth=1,
                    label=f"media = {residuos.mean():.2f}")
    axes[0].set_title("Residuos no tempo", loc="left")
    axes[0].legend(fontsize=8)
    axes[0].tick_params(axis="x", rotation=45)

    # ACF: padrao remanescente que o modelo nao capturou
    plot_acf(residuos, lags=max(1, min(20, len(residuos) // 2 - 1)), ax=axes[1])
    axes[1].set_title("ACF dos residuos", loc="left")

    axes[2].hist(residuos.values, bins=15, color="steelblue", edgecolor="white")
    axes[2].axvline(0, color="black", linewidth=0.8)
    axes[2].set_title("Distribuicao", loc="left")

    fig.suptitle(f"{base_nome} - {modelo_nome}", y=1.04, fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f"Ljung-Box ({lags} lags): estatistica = {stat:.3f} | p-valor = {p:.4f}")
    print("  -> residuos se comportam como ruido branco" if p > 0.05
          else "  -> ainda ha autocorrelacao: sobrou padrao no erro")
    print(f"Vies (media do residuo) = {residuos.mean():.3f} | desvio = {residuos.std():.3f}")

In [ ]:
# melhor modelo de cada base
for base_nome, linha in melhores.iterrows():
    analisar_residuos(base_nome, linha["melhor_modelo"])

## 3.5 Importância das features

A fazer: importância nativa e permutation importance no Random Forest;
coeficientes padronizados e VIP scores no PLS. Destacar onde as externas
aparecem e retomar a disponibilidade delas.